# GeoVision-CLIP Cali — Situacion 3: ConvLSTM + Kriging + LOO-CV

Pipeline completo de deep learning geoespacial y estadistica para predecir
concentraciones de NO2, SO2 y O3 en puntos no muestreados de Cali.

In [ ]:
EMBED_PATH = INPUT / "datasets/edwardsx/geovision-sit3-embeddings"
if not EMBED_PATH.exists():
    EMBED_PATH = INPUT / "geovision-sit3-embeddings"

if EMBED_PATH.exists():
    print("Cargando embeddings desde dataset...")
    data = torch.load(EMBED_PATH / "embeddings_sit3.pt", map_location="cpu", weights_only=True)
    emb = data["embeddings"]
    meta = data["meta"]
    tile_fechas = data["tile_fechas"]
    CACHE_EXISTS = True
else:
    print("Dataset no encontrado. Se generaran embeddings desde cero.")
    CACHE_EXISTS = False


In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

INPUT = Path("/kaggle/input")
PANEL_PATH = INPUT / "datasets/juanjoseorozcolopez/geovision-fuentes"
if not PANEL_PATH.exists():
    PANEL_PATH = INPUT / "geovision-fuentes"

TILES_PATH = INPUT / "datasets/edwardsx/geovision-tiles-sit2"
if not TILES_PATH.exists():
    TILES_PATH = INPUT / "geovision-tiles-sit2"

MODEL_PATH = INPUT / "datasets/edwardsx/geovision-clip-modelo-v2"
if not MODEL_PATH.exists():
    MODEL_PATH = INPUT / "geovision-clip-modelo-v2"

OUTPUT = Path("/kaggle/working")
print(f"Panel: {PANEL_PATH}")
print(f"Tiles: {TILES_PATH}")
print(f"Modelo: {MODEL_PATH}")

CONTAMINANTES = ["NO2", "SO2", "O3"]

# Umbral de cobertura minima para incluir estacion en LOO-CV
COBERTURA_MINIMA = 0.20

# ConvLSTM params
CONV_HIDDEN = 128
CONV_KERNEL = 3
CONV_LAYERS = 2
CONV_LR = 1e-4
CONV_EPOCHS = 30
CONV_BATCH = 16

# Kriging params
KRIGING_VARIOGRAM = "exponential"

## Cargar DAGMA

In [ ]:
pq = pd.read_parquet(PANEL_PATH / "dagma" / "dagma_cvc_horario_raw.parquet")
estaciones = pd.read_csv(PANEL_PATH / "dagma" / "estaciones_metadata.csv")

print(f"Mediciones: {len(pq)}")
print(f"Rango temporal: {pq['med_fecha_inicio'].min()} a {pq['med_fecha_inicio'].max()}")
print(f"Estaciones: {len(estaciones)}")
print()

print("Estaciones y contaminantes:")
tabla = pq.groupby(["nombre_est", "msfl_code"]).size().unstack(fill_value=0)
print(tabla.to_string())


In [ ]:
# Overlap util para Sit 3: 2021-2024 (panel 2021-2025, DAGMA 2020-2024)
pq_util = pq[pq["med_fecha_inicio"].copy() >= "2021-01-01"]
print(f"Mediciones en periodo util (2021-2024): {len(pq_util)}")
print()
print("Cobertura por estacion en periodo util (2021-2024, 48 meses):")
pq_util["ano_mes"] = pq_util["med_fecha_inicio"].dt.to_period("M")
for est in pq_util["nombre_est"].unique():
    mask = pq_util["nombre_est"] == est
    meses = pq_util.loc[mask, "ano_mes"].nunique()
    print(f"  {est:30s} {meses:2d}/48 ({meses/48*100:5.1f}%)")


## Generar secuencias ConvLSTM

In [ ]:
meta = pd.read_parquet(TILES_PATH / "tiles_meta.parquet")
tile_fechas = sorted(set(pd.to_datetime([f[:8] for f in meta["time_s2"].unique()]).date))
print(f"Tiles fechas disponibles: {len(tile_fechas)}")
print(f"Rango: {tile_fechas[0]} a {tile_fechas[-1]}")


In [ ]:
print("Secuencias disponibles por estacion y contaminante:")
total_secuencias = 0
for cont in CONTAMINANTES:
    datos = pq_util[pq_util["msfl_code"] == cont]
    for est in datos["nombre_est"].unique():
        est_datos = datos[datos["nombre_est"] == est].copy()
        est_datos["fecha"] = est_datos["med_fecha_inicio"].dt.date
        fechas_est = sorted(est_datos["fecha"].unique())
        seq = 0
        for f in fechas_est:
            prev = [t for t in tile_fechas if t < f]
            if len(prev) >= 8:
                seq += 1
        if seq > 0:
            print(f"  {est:30s} {cont:3s} {seq:3d} secuencias")
            total_secuencias += seq
print(f"\nTotal secuencias: {total_secuencias}")


## Cargar modelo CLIP + generar embeddings

In [ ]:
import open_clip
from huggingface_hub import hf_hub_download

# Clases LoRA (misma estructura que el entrenamiento)
class LoRALinear(nn.Module):
    def __init__(self, linear, rank=16):
        super().__init__()
        self.linear = linear
        d, k = linear.weight.shape
        self.A = nn.Parameter(torch.randn(d, rank) * 0.01)
        self.B = nn.Parameter(torch.zeros(rank, k))
    @property
    def weight(self): return self.linear.weight
    @property
    def bias(self): return self.linear.bias
    def forward(self, x):
        return self.linear(x) + F.linear(x, self.A @ self.B)

def aplicar_lora(module, rank=16):
    for name, child in module.named_children():
        if isinstance(child, nn.Linear) and name in {"out_proj", "c_fc", "c_proj"}:
            setattr(module, name, LoRALinear(child, rank))
        else:
            aplicar_lora(child, rank)


In [ ]:
ckpt = torch.load(MODEL_PATH / "clip_finetuned_best.pt", map_location="cpu", weights_only=True)

clip_model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained=None)

# Adaptar conv1 3ch -> 12ch
orig = clip_model.visual.conv1
new_conv = nn.Conv2d(12, orig.out_channels, orig.kernel_size, stride=orig.stride, bias=False)
with torch.no_grad():
    w = new_conv.weight.data
    w[:, 0] = orig.weight[:, 1]  # B4 -> R
    w[:, 1] = orig.weight[:, 2]  # B3 -> G
    w[:, 2] = orig.weight[:, 0]  # B2 -> B
    for b in range(3, 12):
        w[:, b] = orig.weight.mean(dim=1) * (3.0 / 12)
clip_model.visual.conv1 = new_conv

# Aplicar LoRA
aplicar_lora(clip_model.visual.transformer.resblocks[6:])
aplicar_lora(clip_model.transformer.resblocks[6:])

# Cargar pesos
clip_model.load_state_dict(ckpt["clip"], strict=False)
clip_model = clip_model.to(DEVICE).eval()

# Fusion
class VisualProj(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(512, 512)
    def forward(self, x): return self.proj(x)
fusion = VisualProj()
fusion.load_state_dict(ckpt["fusion"])
fusion = fusion.to(DEVICE).eval()

print("Modelo + Fusion cargados")


In [ ]:
IDX_OPTICAS = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])
TILE_PX_CLIP = 224
BATCH = 64

# Normalizacion bandas
tiles_arr = np.load(TILES_PATH / "tiles_train.npz", allow_pickle=False)["data"]
flat = tiles_arr[:, IDX_OPTICAS].reshape(len(tiles_arr), 12, -1)
BAND_MEAN = flat.mean(axis=(0, 2)).astype(np.float32)
BAND_STD = flat.std(axis=(0, 2)).astype(np.float32) + 1e-6

@torch.no_grad()
def generar_embeddings():
    embs = []
    for i in tqdm(range(0, len(tiles_arr), BATCH), desc="Embeddings"):
        x = tiles_arr[i:i+BATCH, IDX_OPTICAS].astype(np.float32)
        x = (x - BAND_MEAN[None, :, None, None]) / BAND_STD[None, :, None, None]
        x = np.clip(x, -3.0, 3.0)
        x = torch.from_numpy(x).to(DEVICE)
        x = F.interpolate(x, size=TILE_PX_CLIP, mode="bilinear", align_corners=False)
        fused = F.normalize(fusion(clip_model.encode_image(x)), dim=-1)
        embs.append(fused.cpu())
    return torch.cat(embs)

emb = generar_embeddings()
print(f"Embeddings: {emb.shape}")


## Construir secuencias ConvLSTM

In [ ]:
meta["fecha_dt"] = pd.to_datetime(meta["time_s2"].astype(str).str.split("_").str[0], format="%Y%m%dT%H%M%S")
tile_fechas = sorted(set(meta["fecha_dt"].dt.date))
idx_por_fecha = {f: meta[meta["fecha_dt"].dt.date == f].index.values for f in tile_fechas}
print(f"Fechas tile: {len(tile_fechas)}, Embeddings: {emb.shape}")


In [ ]:
from collections import defaultdict

secuencias = defaultdict(list)
contador = 0

for cont in CONTAMINANTES:
    datos_cont = pq_util[pq_util["msfl_code"] == cont]
    for est in datos_cont["nombre_est"].unique():
        est_datos = datos_cont[datos_cont["nombre_est"] == est].copy()
        est_datos["fecha"] = est_datos["med_fecha_inicio"].dt.date
        for fecha, grupo in est_datos.groupby("fecha"):
            prev = [t for t in tile_fechas if t < fecha]
            if len(prev) >= 8:
                ultimas_8 = prev[-8:]
                # Verificar que haya tiles para esas 8 fechas
                hay_tiles = all(len(idx_por_fecha.get(f, [])) > 0 for f in ultimas_8)
                if hay_tiles:
                    contador += 1
                    if contador <= 5:
                        print(f"  {est:30s} {cont:3s} fecha={fecha} -> 8 tiles prev: {ultimas_8[0]} a {ultimas_8[-1]}")

print(f"\nTotal secuencias validas: {contador}")


In [ ]:
torch.save({
    "embeddings": emb.cpu(),
    "meta": meta,
    "tile_fechas": tile_fechas,
}, OUTPUT / "embeddings_sit3.pt")
print(f"Embeddings guardados: {OUTPUT / 'embeddings_sit3.pt'}")
print(f"  embeddings: {emb.shape}")
print(f"  meta rows: {len(meta)}")


In [ ]:
model = ConvLSTM(input_dim=512, hidden_dim=128, kernel_size=3, num_layers=2).to(DEVICE)
print(f"ConvLSTM: {sum(p.numel() for p in model.parameters())/1024:.0f}K params")


## Preparar datos reales + entrenamiento

In [ ]:
meta["fecha_dt"] = pd.to_datetime(meta["time_s2"].astype(str).str.split("_").str[0], format="%Y%m%dT%H%M%S")
tile_fechas = sorted(set(meta["fecha_dt"].dt.date))
idx_por_fecha = {f: meta[meta["fecha_dt"].dt.date == f].index.values for f in tile_fechas}

X_seqs, y_vals = [], []
for cont in CONTAMINANTES:
    datos_cont = pq_util[pq_util["msfl_code"] == cont]
    for est in datos_cont["nombre_est"].unique():
        sub = datos_cont[datos_cont["nombre_est"] == est]
        sub["fecha"] = sub["med_fecha_inicio"].dt.date
        for fecha, grupo in sub.groupby("fecha"):
            prev = [t for t in tile_fechas if t < fecha]
            if len(prev) >= 8:
                idxs = [idx_por_fecha[f][0] for f in prev[-8:] if len(idx_por_fecha.get(f,[]))>0]
                if len(idxs) == 8:
                    X_seqs.append(emb[idxs].numpy())
                    y_vals.append(grupo["med_concentracion_estandar"].mean())
print(f"Dataset: {len(X_seqs)} secuencias")


In [ ]:
X_t = torch.FloatTensor(np.array(X_seqs)).unsqueeze(-1).unsqueeze(-1)
y_t = torch.FloatTensor(y_vals)
dataset = torch.utils.data.TensorDataset(X_t, y_t)
loader = DataLoader(dataset, batch_size=16, shuffle=True)
model.train()
optim = torch.optim.AdamW(model.parameters(), lr=1e-4)
for ep in range(10):
    total = 0
    for xb, yb in loader:
        pred = model(xb.to(DEVICE))[:, :, 0, 0]
        loss = F.mse_loss(pred, yb.to(DEVICE).unsqueeze(1).expand(-1, 3))
        optim.zero_grad(); loss.backward(); optim.step()
        total += loss.item()
    print(f"Epoch {ep+1:2d}: loss={total/len(loader):.4f}")


In [ ]:
print("\n=== Kriging Ordinario LOO-CV ===\n")

for cont in ["SO2", "O3"]:
    datos_cont = pq_util[pq_util["msfl_code"] == cont]
    estaciones = datos_cont["nombre_est"].unique()
    lats, lons, vals, ests = [], [], [], []

    for est in estaciones:
        sub = datos_cont[datos_cont["nombre_est"] == est]
        if len(sub) > 10:
            lats.append(sub["latitud"].mean())
            lons.append(sub["longitud"].mean())
            vals.append(sub["med_concentracion_estandar"].mean())
            ests.append(est)

    lats = np.array(lats); lons = np.array(lons); vals = np.array(vals)
    print(f"\n{cont}: {len(lats)} estaciones")

    errores = []
    for i in range(len(lats)):
        mascara = np.ones(len(lats), dtype=bool)
        mascara[i] = False
        lat_test, lon_test = lats[i], lons[i]
        lat_train, lon_train = lats[mascara], lons[mascara]
        val_real = vals[i]
        val_train = vals[mascara]

        if len(lat_train) < 3:
            continue

        try:
            ok = OrdinaryKriging(lat_train, lon_train, val_train,
                                 variogram_model="exponential",
                                 verbose=False, enable_statistics=False)
            pred, _ = ok.execute("points", [lat_test], [lon_test])
            errores.append(abs(pred[0] - val_real))
            print(f"  {ests[i]:30s} real={val_real:6.2f} pred={pred[0]:6.2f} error={abs(pred[0]-val_real):.2f}")
        except Exception as e:
            print(f"  {ests[i]:30s} ERROR: {e}")

    if len(errores) > 0:
        print(f"  MAE Kriging: {np.mean(errores):.2f} ug/m3")
